# Análise Exploratória dos Dados

Imports padrão:

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

Download do Dataset:

In [ ]:
import os
import requests

url = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2026/INFLUD26-19-10-2026.parquet"
local_filename = "../data/srag_data_2026.parquet"

if os.path.exists(local_filename):
    print("Arquivo já existe, download ignorado.")
else:
    os.makedirs(os.path.dirname(local_filename), exist_ok=True)

    # download e escrita
    response = requests.get(url, allow_redirects=True)
    response.raise_for_status()  # HTTP errors

    with open(local_filename, "wb") as file:
        file.write(response.content)

    print("Download completo.")

Carregamento dos dados:

In [ ]:
import pandas as pd
df = pd.read_parquet(path=local_filename)

## Descrição dos dados

Formato dos dados:

In [ ]:
df.shape

Temos ~232 mil registros e 194 colunas. Devido ao número de colunas, o formato `Parquet` é uma boa escolha: por ser colunar, permite carregar em memória somente as colunas necessárias (parâmetro `columns` de `pd.read_parquet`). Aqui carregamos o arquivo inteiro uma única vez para explorá-lo e depois o reduzimos aos campos de interesse. Aplicando um breve `pd.DataFrame.describe()` temos:

In [ ]:
df.describe().columns

Vemos que somente as colunas de datas aparecem no resumo, pois não há colunas numéricas: de acordo com o [2019 a 2025 - Dicionário de Dados](https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026/resource/3135ac9c-2019-4989-a893-2ed50ebd8e68), os campos são datas ou textos (`varchar`):

In [ ]:
df.info(verbose=True)

Vemos um dataset relativamente grande (~400 MB; a depender da sua memória, no meu caso escassa), logo será útil usarmos somente as colunas necessárias. Com base no dicionário de dados, as colunas úteis para responder às nossas perguntas são (ver [2019 a 2025 - Dicionário de Dados](https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026/resource/3135ac9c-2019-4989-a893-2ed50ebd8e68)):

| Nome da coluna | Descrição | Tipo | Obrigatório? |
| --- | --- | --- | --- |
| `NU_NOTIFIC` | Número de registro sequencial gerado automaticamente pelo sistema. O dígito 1 caracteriza o tipo da ficha (1=SG, 2=SRAG-UTI e 3=SRAG Hospitalizado) e os dígitos 2 a 12 são gerados automaticamente pelo sistema | `str` | ✅ |
| `DT_NOTIFIC` | Data de preenchimento da ficha de notificação. | `datetime64[ns]` | ✅ |
| `DT_SIN_PRI` | Data de 1º sintomas do caso. | `datetime64[ns]` | ✅ |
| `DT_INTERNA` | Data em que o paciente foi hospitalizado. | `datetime64[ns]` | ✅ |
| `DT_EVOLUCA` | Data da alta ou óbito. | `datetime64[ns]` | ❌ |
| `EVOLUCAO` | Evolução do caso (1-Cura / 2-Óbito / 3-Óbito por outras causas / 9-Ignorado). | `str` | ❌ |
| `VACINA_COV` | Informar se o paciente recebeu vacina COVID-19, após verificar a documentação/caderneta (1-Sim / 2-Não / 9-Ignorado). | `str` | ✅ |
| `UTI` | Indica se o paciente foi internado em UTI (1-Sim / 2-Não / 9-Ignorado). | `str` | ❌ |

Para calcularmos as métricas:
- Taxa de aumento de casos;
- Taxa de mortalidade;
- Taxa de ocupação de UTI (proporção dos casos internados em UTI);
- Taxa de vacinação (proporção de vacinados entre os casos);
- Número diário de casos dos últimos 30 dias;
- Número mensal de casos dos últimos 12 meses.

Os dados reduzidos somente a estas colunas têm a seguinte forma:

In [ ]:
cols = ["NU_NOTIFIC", "DT_NOTIFIC", "DT_SIN_PRI", "DT_INTERNA", "DT_EVOLUCA", "EVOLUCAO", "VACINA_COV", "UTI"]
df_reduced = df[cols].copy()
df_reduced.head(10)

Para simplificar a análise, vamos renomear alguns rótulos:

In [ ]:
evolucao = {"1":"cura", "2":"obito", "3":"obito"}
vacinacao = {"1":"vacinado", "2":"nao_vacinado", "9": pd.NA}
uti = {"1":"uti", "2":"nao_uti", "9": pd.NA}
df_reduced["EVOLUCAO"] = df_reduced["EVOLUCAO"].map(evolucao)
df_reduced["VACINA_COV"] = df_reduced["VACINA_COV"].map(vacinacao)
df_reduced["UTI"] = df_reduced["UTI"].map(uti)

Note que juntamos óbitos e óbitos por outras causas em uma única categoria (`obito`) e tratamos os valores ignorados como nulos:

In [ ]:
df_reduced.head(15)

Vejamos os valores nulos:

In [ ]:
df_reduced.isna().sum() * 100 / len(df_reduced)

Precisamos das colunas `EVOLUCAO`, `VACINA_COV` e `UTI` para calcular as taxas de mortalidade, vacinação e internação em UTI, respectivamente. Sendo assim, retirando os nulos, temos:

In [ ]:
df_cleaned = df_reduced.dropna(subset=["EVOLUCAO", "VACINA_COV", "UTI"])
df_cleaned.isna().sum() * 100 / len(df_cleaned)

A porcentagem de registros descartados é:

In [ ]:
(len(df_reduced) - len(df_cleaned)) * 100 / len(df_reduced)

Perdemos ~23% dos dados e vamos aceitar essa perda. Note que isso introduz um viés: casos ainda sem desfecho (`EVOLUCAO` nulo) na data de extração são descartados, o que tende a subestimar o número de casos mais recentes. Agora, com o dataset limpo, vamos analisar cada etapa.

## Taxa de aumento de casos e número de casos

Para analisar o número de casos e a taxa de aumento, vamos decidir primeiro qual data usar: data de notificação (`DT_NOTIFIC`) ou data dos primeiros sintomas (`DT_SIN_PRI`). Para isso, criamos séries temporais do número de casos considerando as duas datas:

In [ ]:
# usando data dos primeiros sintomas
num_casos_notificado_dt_pri_sin = df_cleaned.groupby('DT_SIN_PRI')['NU_NOTIFIC'].count()
# usando datas dos casos de notificação
num_casos_notificado_dt_not = df_cleaned.groupby('DT_NOTIFIC')['NU_NOTIFIC'].count()

Podemos calcular a correlação entre as duas séries:

In [ ]:
num_casos_notificado_dt_not.corr(num_casos_notificado_dt_pri_sin)

As séries são moderadamente correlacionadas; no gráfico vemos:

In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_notificado_dt_pri_sin, marker="o", label = "Primeiros Sintomas", alpha=0.8)
sns.lineplot(data=num_casos_notificado_dt_not, marker="d", label = "Data de Notificação", alpha=0.8)

plt.title("Séries temporais do número de casos")
plt.xlabel("Data")
plt.ylabel("Número de casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Vemos uma periodicidade no número de casos por data de notificação: ao fim da semana o número de casos decai, sendo o provável motivo a ausência de notificações nesses dias. Matematicamente, podemos ver através de uma transformada de Fourier que o pico de períodos ocorre por volta de 7 dias, estabelecendo assim uma certa periodicidade nos dados (parafraseando livros-texto de matemática, a prova fica para o leitor). Sendo assim, podemos usar a média móvel semanal (centrada) do número de casos por data de notificação. A correlação dessa média móvel com a série por data de primeiros sintomas é:

In [ ]:
num_casos_notificado_dt_not_mov_avg = num_casos_notificado_dt_not.rolling(7, center=True).mean()
num_casos_notificado_dt_not_mov_avg.corr(num_casos_notificado_dt_pri_sin)

Como as séries são altamente correlacionadas, o uso das duas é redundante e vamos usar somente a série temporal por data de primeiros sintomas. Comparemos as séries, agora com a média móvel, só por descargo de consciência:

In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_notificado_dt_pri_sin, marker="o", label = "Primeiros Sintomas", alpha=0.8)
sns.lineplot(data=num_casos_notificado_dt_not_mov_avg, marker="d", label = "Data de Notificação (média móvel)", alpha=0.8)

plt.title("Séries temporais do número de casos")
plt.xlabel("Data")
plt.ylabel("Número de casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Calculemos agora o número de casos nos últimos 30 dias e 12 meses. Como o dataset é um retrato estático, consideramos a data mais recente de primeiros sintomas (`max_date`) como "hoje". Note que, com o arquivo de 2026 apenas, a janela de 12 meses cobre menos de 12 meses reais. Os conjuntos de dados nesses períodos são:

In [ ]:
max_date = df_cleaned["DT_SIN_PRI"].max()
initial_30_d = max_date - pd.Timedelta(days=30)
initial_12_m = max_date - pd.Timedelta(days=365)

df_reduced_30_d = df_cleaned[df_cleaned["DT_SIN_PRI"]>initial_30_d]
df_reduced_12_m = df_cleaned[df_cleaned["DT_SIN_PRI"]>initial_12_m]

num_casos_30_d = df_reduced_30_d.groupby('DT_SIN_PRI')['NU_NOTIFIC'].count()
num_casos_12_m = df_reduced_12_m.groupby('DT_SIN_PRI')['NU_NOTIFIC'].count()

Sendo assim:

In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_30_d, marker="o", alpha=0.8)

plt.title("Número de casos últimos 30 dias")
plt.xlabel("Data")
plt.ylabel("Número de casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_12_m, marker="o", alpha=0.8)

plt.title("Número de casos últimos 12 meses")
plt.xlabel("Data")
plt.ylabel("Número de casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Para o crescimento do número de casos vamos utilizar a taxa mês a mês. Primeiro fazemos o *resample* mensal:

In [ ]:
num_casos_mensal = num_casos_12_m.resample('ME').sum()
num_casos_mensal

Em seguida calculamos a variação percentual em relação ao mês anterior. **Atenção:** o último mês é parcial (neste dataset, outubro vai somente até 16/10), então sua variação, fortemente negativa, reflete em grande parte a janela incompleta e não uma queda real. Pelo mesmo motivo, o primeiro mês da janela também pode ser parcial em outros datasets.

In [ ]:
crescimento_mensal_num_casos = num_casos_mensal.pct_change().dropna()*100
crescimento_mensal_num_casos

Outras estatísticas (o mínimo coincide com o último dia do dataset, provavelmente por atraso de notificação):

In [ ]:
max_num_casos_30_d = num_casos_30_d.idxmax(), num_casos_30_d.max()
min_num_casos_30_d = num_casos_30_d.idxmin(), num_casos_30_d.min()

max_num_casos_12_m = num_casos_12_m.idxmax(), num_casos_12_m.max()
min_num_casos_12_m = num_casos_12_m.idxmin(), num_casos_12_m.min()

std_num_casos_30_d = num_casos_30_d.std()
std_num_casos_12_m = num_casos_12_m.std()

mean_num_casos_30_d = num_casos_30_d.mean()
mean_num_casos_12_m = num_casos_12_m.mean()

print("Últimos 30 dias:")
print("Data do máximo de casos:", max_num_casos_30_d[0], "Número:", max_num_casos_30_d[1])
print("Data do mínimo de casos:", min_num_casos_30_d[0], "Número:", min_num_casos_30_d[1])
print("Desvio padrão:", std_num_casos_30_d)
print("Média:", mean_num_casos_30_d)
print("")
print("Últimos 12 meses:")
print("Data do máximo de casos:", max_num_casos_12_m[0], "Número:", max_num_casos_12_m[1])
print("Data do mínimo de casos:", min_num_casos_12_m[0], "Número:", min_num_casos_12_m[1])
print("Desvio padrão:", std_num_casos_12_m)
print("Média:", mean_num_casos_12_m)

## Taxa de vacinação

Vejamos agora a proporção de casos de SRAG vacinados contra COVID-19 (não é a cobertura vacinal da população geral):

In [ ]:
vacinados_clean = df_cleaned.groupby("VACINA_COV").count()[["NU_NOTIFIC"]]
vacinados_clean["pct"] = vacinados_clean["NU_NOTIFIC"]*100/len(df_cleaned)
vacinados_clean

## Taxa de mortalidade

Últimos 30 dias:

In [ ]:
mortalidade_clean_30_d = df_reduced_30_d.groupby("EVOLUCAO").count()[["NU_NOTIFIC"]]
mortalidade_clean_30_d["pct"] = mortalidade_clean_30_d["NU_NOTIFIC"] * 100/len(df_reduced_30_d)
mortalidade_clean_30_d

Últimos 12 meses:

In [ ]:
mortalidade_clean_12_m = df_reduced_12_m.groupby("EVOLUCAO").count()[["NU_NOTIFIC"]]
mortalidade_clean_12_m["pct"] = mortalidade_clean_12_m["NU_NOTIFIC"] * 100/len(df_reduced_12_m)
mortalidade_clean_12_m

## Taxa de internação em UTI

Últimos 30 dias:

In [ ]:
internacao_clean_30_d = df_reduced_30_d.groupby("UTI").count()[["NU_NOTIFIC"]]
internacao_clean_30_d["pct"] = internacao_clean_30_d["NU_NOTIFIC"] * 100/len(df_reduced_30_d)
internacao_clean_30_d

Últimos 12 meses:

In [ ]:
internacao_clean_12_m = df_reduced_12_m.groupby("UTI").count()[["NU_NOTIFIC"]]
internacao_clean_12_m["pct"] = internacao_clean_12_m["NU_NOTIFIC"] * 100/len(df_reduced_12_m)
internacao_clean_12_m